In [1]:
TOKEN=$(curl -s -X POST http://127.0.0.1:8000/api/auth/login -H "Content-Type: application/json" -d "{\"username\":\"testuser\",\"password\":\"testpass123\"}" 

SyntaxError: invalid syntax (887983586.py, line 1)

In [1]:
import sys, json
data = json.load(sys.stdin)
ids = [a.get('id') for a in data['articles']]
print('返却件数:', len(ids))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [2]:
from datetime import datetime, timezone
from db import crud

now_str = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")

TEST_LINK = "https://example.com/test-duplicate-article-zzz"
TEST_TITLE = "TEST_DUPLICATE_ARTICLE_ZZZ_12345"

crud.save_article_as_duplicate({
    "link":           TEST_LINK,
    "title_original": TEST_TITLE,
    "title_ja":       TEST_TITLE,
    "summary_ja":     None,
    "category":       None,
    "is_spurs":       0,
    "has_score":      0,
    "is_duplicate":   1,
    "score_data":     None,
    "source":         "test",
    "published_at":   now_str,
    "fetched_at":     now_str,
})

print("テストデータ投入完了:")
print("  link =", TEST_LINK)
print("  title_original =", TEST_TITLE)

テストデータ投入完了:
  link = https://example.com/test-duplicate-article-zzz
  title_original = TEST_DUPLICATE_ARTICLE_ZZZ_12345


In [3]:
import requests

BASE = "http://127.0.0.1:8000/api"
TEST_TITLE = "TEST_DUPLICATE_ARTICLE_ZZZ_12345"

res = requests.post(f"{BASE}/auth/login", json={
    "username": "testuser",
    "password": "testpass123",
})
token = res.json()["access_token"]
headers = {"Authorization": f"Bearer {token}"}

res = requests.get(f"{BASE}/news?limit=100", headers=headers)
articles = res.json()["articles"]

titles = [a["title_original"] for a in articles]

print("status:", res.status_code)
print("取得件数:", len(articles))
print("テスト記事がレスポンスに含まれているか:", TEST_TITLE in titles)

status: 200
取得件数: 100
テスト記事がレスポンスに含まれているか: False


In [4]:
from db.models import SessionLocal, Article

with SessionLocal() as session:
    row = session.query(Article).filter_by(
        link="https://example.com/test-duplicate-article-zzz"
    ).first()
    print("DB保存確認:", row is not None)
    print("is_duplicate値:", row.is_duplicate if row else None)

DB保存確認: True
is_duplicate値: 1


In [5]:
from db.models import SessionLocal, Article

with SessionLocal() as session:
    session.query(Article).filter_by(
        link="https://example.com/test-duplicate-article-zzz"
    ).delete()
    session.commit()
    print("テストデータ削除完了")

テストデータ削除完了
